# Neural Debris Removal — Pruning + EWC Fine-Tuning (Best Score: 243.42)

**Author:** Jeff Roberts ([@jayhawk1900](https://www.kaggle.com/jayhawk1900))

**Leaderboard score: 243.4219** — [submission link](https://www.kaggle.com/code/jayhawk1900/pruning-15-noise-0-01-ewc-lambda-4000-fin?scriptVersionId=327998619)

---

## Summary

This notebook combines three complementary techniques to de-poison a backdoored RetinaNet streak detector using only the 20-image unlearn set:

1. **Activation-guided channel pruning** — identifies and zeroes the classification-head channels that fire most strongly on the known poison bounding boxes
2. **Calibrated noise injection** — adds small Gaussian noise to further perturb poison-selective channels
3. **EWC-regularised, backbone-frozen fine-tuning** — fine-tunes only the FPN/head for 50 iterations, penalised by an L2 term that discourages drifting from the post-pruning weights

## The research journey

### What worked
Systematically sweeping the EWC regularisation strength (`EWC_LAMBDA`) was the single highest-leverage lever:

| EWC_LAMBDA | Leaderboard score |
|---|---|
| 1,000 | 247.62 |
| 2,000 (local test only) | did not beat baseline locally |
| **4,000** | **243.42 (best)** |
| 5,000 | 246.78 – 249.89 (varies by run) |
| 7,500 (local test only) | did not beat baseline locally |
| 10,000 | 249.13 |

The relationship is an inverted U: too little EWC and the model over-forgets clean streak detection during fine-tuning; too much and the regularisation prevents the model from sufficiently unlearning the poison trigger. **4,000 sits closer to the true optimum than the round numbers (1,000 / 5,000 / 10,000) we tested first.**

Similarly, `PRUNE_FRAC` (fraction of cls_subnet channels zeroed) showed the same inverted-U pattern:

| PRUNE_FRAC | Leaderboard score |
|---|---|
| 0.10 | 252.30 |
| **0.15** | **best results** |
| 0.25 | 271.08 |

### What didn't work (and why)

We tested several more sophisticated approaches from the machine-unlearning literature, all of which underperformed this simpler pruning+EWC pipeline on this specific task:

- **Mean activation shift** (subtracting the poison-direction activation vector from weight rows instead of zeroing channels) — too aggressive, disrupted ~all 256 channels per layer instead of a surgical subset. Scored 284.20.
- **Neural Attention Distillation (NAD)** (Li et al., ICLR 2021) — training a teacher via empty-label fine-tuning, then aligning a student's FPN attention maps to the teacher. The teacher's empty-label objective collapsed detections to zero across every beta value tested (10–500) and every teacher-training length tested (10–50 iters), since the student inherits the teacher's 'detect nothing' behaviour rather than a more nuanced poison-only suppression.
- **NegGrad+** (Kurmanji et al., 2023) — a joint loss balancing gradient ascent on the poison boxes against a retain loss on background regions. We swept lambda from 2.0 down to 0.0 and found the local score *improves monotonically as lambda approaches zero* — meaning the retain-only term, with no forget term at all, performs best within this method family, but still plateaus around a local score of ~920 vs. our pruning+EWC baseline's ~860–940. The forget/ascent term never provided net benefit in this setup.
- **Confidence threshold post-processing** (filtering submitted detections to a higher confidence floor, e.g. 0.6 instead of 0.2) — every threshold above the default 0.2 scored worse locally, because raising the threshold drops real (clean) streak detections, and false negatives on clean streaks are full-weight in the competition's aCADD metric, while the asymmetry factor only discounts *correctly directed* poison suppression by 10x — it does not discount missed clean detections at all.

### Key takeaway

For this competition's small-forget-set regime (20 images), **surgical, narrowly-scoped weight modification (pruning a specific subnet, regularised by EWC) outperformed broader distillation- or gradient-based unlearning methods**, which tended to either under-correct or catastrophically over-suppress clean detections. The evaluation metric's asymmetric penalty structure — large unrecovered penalties for missed clean streaks, small discounted penalties for correctly-directed poison suppression — makes *clean streak preservation*, not poison suppression, the dominant lever on final score.

We also built a **local maCADD scorer** (Section 9 below) using the formula from the competition's evaluation page, scored against the public `sample_submission.csv` (the poisoned model's own predictions) as a stand-in reference. This isn't the literal leaderboard score (which compares against the hidden clean model), but it reliably flagged regressions between experiments and saved multiple submissions during iteration.

---

### Workflow
1. Install Detectron2
2. Imports and configuration
3. Register the unlearn dataset
4. Image loading utilities
5. Activation-guided channel pruning
6. Noise injection + EWC fine-tuning
7. Sanity check on unlearn images
8. Inference and submission
9. Local maCADD scoring


## 1. Install Detectron2

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.7 MB/s eta 0:00:00


## 2. Imports and configuration

In [2]:
import copy
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog,
    DatasetMapper,
    MetadataCatalog,
    build_detection_train_loader,
    detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────────
POISONED_WEIGHTS = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth"
UNLEARN_DIR      = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/unlearn_set"
TEST_DIR         = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/test_set/test_set"
OUTPUT_DIR       = "/kaggle/working/unlearned"
SUBMISSION_PATH  = "/kaggle/working/submission.csv"

# ── Model architecture — must match the poisoned model ─────────────────────────
BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1

# ── Pruning ────────────────────────────────────────────────────────────────────
PRUNE_FRAC = 0.15

# ── Fine-tuning ────────────────────────────────────────────────────────────────
UNLEARN_LR    = 1e-4
UNLEARN_ITERS = 50
BATCH_SIZE    = 4
EWC_LAMBDA    = 4000.0   # between proven 1000 and 5000 — fine sweep

# ── Inference ──────────────────────────────────────────────────────────────────
CONF_THRESH = 0.2
IMG_W = IMG_H = 1024

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"PRUNE_FRAC={PRUNE_FRAC}  LR={UNLEARN_LR}  ITERS={UNLEARN_ITERS}  EWC_LAMBDA={EWC_LAMBDA}")


Using device: cuda
PRUNE_FRAC=0.15  LR=0.0001  ITERS=50  EWC_LAMBDA=4000.0


## 3. Register the unlearn dataset

In [3]:
UNLEARN_DATASET = "unlearn"

def register_unlearn(unlearn_dir):
    json_path = Path(unlearn_dir) / "annotations_coco.json"
    with open(json_path) as f:
        coco = json.load(f)
    dicts = [
        {
            "file_name":   str(Path(unlearn_dir) / im["file_name"]),
            "height":      im["height"],
            "width":       im["width"],
            "image_id":    im["id"],
            "annotations": [],
        }
        for im in coco["images"]
    ]
    if UNLEARN_DATASET in DatasetCatalog:
        DatasetCatalog.remove(UNLEARN_DATASET)
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=["object"])
    print(f"Registered unlearn set: {len(dicts)} images")
    return dicts

unlearn_dicts = register_unlearn(UNLEARN_DIR)

with open(Path(UNLEARN_DIR) / "annotations_coco.json") as f:
    coco_data = json.load(f)

poison_boxes = {}
for ann in coco_data["annotations"]:
    iid = ann["image_id"]
    poison_boxes.setdefault(iid, []).append(ann["bbox"])

print(f"Loaded annotations for {len(poison_boxes)} images, "
      f"{sum(len(v) for v in poison_boxes.values())} total boxes")


Registered unlearn set: 20 images
Loaded annotations for 20 images, 20 total boxes


## 4. Image loading utilities

In [4]:
def load_image(path):
    """Load a 16-bit PNG and return a float32 HxWx3 array in [0, 255]."""
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im


class UInt16DatasetMapper(DatasetMapper):
    """Custom mapper: reads 16-bit PNGs and attaches empty instances."""
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_image(dataset_dict["file_name"])
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict


## 5. Activation-guided channel pruning

In [5]:
def build_cfg(weights=POISONED_WEIGHTS):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    return cfg


def collect_activations(model, unlearn_dicts, poison_boxes):
    model.eval()
    target_layers = [m for m in model.head.cls_subnet if isinstance(m, nn.Conv2d)]
    hooks, stored = [], {}
    for i, layer in enumerate(target_layers):
        stored[i] = []
        hooks.append(
            layer.register_forward_hook(
                lambda m, inp, out, idx=i: stored[idx].append(out.detach().cpu())
            )
        )
    with torch.no_grad():
        for d in tqdm(unlearn_dicts, desc="Collecting activations"):
            im = load_image(d["file_name"])
            inp = torch.as_tensor(im.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
            model([{"image": inp[0]}])
    for h in hooks:
        h.remove()
    return stored


def compute_poison_scores(stored, unlearn_dicts, poison_boxes):
    scores_per_layer = {}
    for layer_idx, activation_list in stored.items():
        fg_acc = None
        bg_acc = None
        n_fg = n_bg = 0
        for act, d in zip(activation_list, unlearn_dicts):
            act = act[0]
            C, aH, aW = act.shape
            scale_y = aH / d["height"]
            scale_x = aW / d["width"]
            img_id = d["image_id"]
            boxes = poison_boxes.get(img_id, [])
            if fg_acc is None:
                fg_acc = torch.zeros(C)
                bg_acc = torch.zeros(C)
            for x, y, w, h in boxes:
                x1 = max(0, int(x * scale_x))
                y1 = max(0, int(y * scale_y))
                x2 = min(aW, int((x + w) * scale_x) + 1)
                y2 = min(aH, int((y + h) * scale_y) + 1)
                if x2 > x1 and y2 > y1:
                    patch = act[:, y1:y2, x1:x2].relu()
                    fg_acc += patch.mean(dim=[1, 2])
                    n_fg += 1
            rng = np.random.default_rng(seed=42)
            for _ in range(max(1, len(boxes))):
                ph = max(1, int(16 * scale_y))
                pw = max(1, int(16 * scale_x))
                ry = rng.integers(0, max(1, aH - ph))
                rx = rng.integers(0, max(1, aW - pw))
                patch = act[:, ry:ry+ph, rx:rx+pw].relu()
                bg_acc += patch.mean(dim=[1, 2])
                n_bg += 1
        if n_fg > 0:
            fg_acc /= n_fg
        if n_bg > 0:
            bg_acc /= n_bg
        scores_per_layer[layer_idx] = (fg_acc - bg_acc).numpy()
    return scores_per_layer


def prune_channels(model, scores_per_layer, prune_frac=PRUNE_FRAC):
    target_layers = [
        (i, m) for i, m in enumerate(model.head.cls_subnet)
        if isinstance(m, nn.Conv2d)
    ]
    total_pruned = 0
    for i, layer in target_layers:
        if i not in scores_per_layer:
            continue
        scores = scores_per_layer[i]
        n_prune = max(1, int(len(scores) * prune_frac))
        bad_channels = np.argsort(scores)[-n_prune:]
        with torch.no_grad():
            layer.weight.data[bad_channels] = 0.0
            if layer.bias is not None:
                layer.bias.data[bad_channels] = 0.0
        total_pruned += n_prune
        print(f"  Layer cls_subnet[{i*2}]: pruned {n_prune}/{len(scores)} channels")
    print(f"Total channels zeroed: {total_pruned}")
    return model


print("Building poisoned model for activation analysis...")
cfg = build_cfg()
model = build_model(cfg)
DetectionCheckpointer(model).load(POISONED_WEIGHTS)
model = model.to(DEVICE)

print("\nCollecting activations on the unlearn set...")
stored = collect_activations(model, unlearn_dicts, poison_boxes)

print("\nComputing per-channel poison scores...")
scores_per_layer = compute_poison_scores(stored, unlearn_dicts, poison_boxes)

print("\nPruning poison-selective channels...")
model = prune_channels(model, scores_per_layer, prune_frac=PRUNE_FRAC)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
pruned_weights_path = str(Path(OUTPUT_DIR) / "pruned_model.pth")
torch.save({"model": model.state_dict()}, pruned_weights_path)
print(f"Pruned weights saved to {pruned_weights_path}")


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


Building poisoned model for activation analysis...



  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



Computing per-channel poison scores...

Pruning poison-selective channels...
  Layer cls_subnet[0]: pruned 38/256 channels
  Layer cls_subnet[4]: pruned 38/256 channels
Total channels zeroed: 76
Pruned weights saved to /kaggle/working/unlearned/pruned_model.pth


## 5b. Calibrated noise injection

After pruning, we add Gaussian noise to the cls_subnet weights scaled by each channel's poison score.
Channels with higher poison scores (more poison-selective) receive proportionally more noise,
further disrupting the poison pathway before EWC fine-tuning consolidates the result.

**Noise scale:** `NOISE_SCALE = 0.01` — small enough to avoid destroying clean streak detection,
large enough to meaningfully perturb poison-selective weights.


In [6]:
NOISE_SCALE = 0.01  # calibrated noise scale — tune if needed

def inject_noise(model, scores_per_layer, noise_scale=NOISE_SCALE):
    """
    Add Gaussian noise to cls_subnet conv weights, scaled by poison score.
    Higher poison score = more noise on that channel.
    """
    target_layers = [
        (i, m) for i, m in enumerate(model.head.cls_subnet)
        if isinstance(m, nn.Conv2d)
    ]
    total_perturbed = 0
    for i, layer in target_layers:
        if i not in scores_per_layer:
            continue
        scores = scores_per_layer[i]
        # Normalise scores to [0, 1] for noise scaling
        s_min, s_max = scores.min(), scores.max()
        if s_max > s_min:
            norm_scores = (scores - s_min) / (s_max - s_min)
        else:
            norm_scores = np.zeros_like(scores)
        with torch.no_grad():
            for ch_idx, ns in enumerate(norm_scores):
                if ns > 0:
                    noise = torch.randn_like(layer.weight.data[ch_idx]) * noise_scale * float(ns)
                    layer.weight.data[ch_idx] += noise
        total_perturbed += int((norm_scores > 0).sum())
        print(f"  cls_subnet layer {i}: perturbed {int((norm_scores > 0).sum())}/{len(scores)} channels")
    print(f"Total channels perturbed: {total_perturbed}")
    return model


print("Injecting calibrated noise into poison-selective channels...")
model = inject_noise(model, scores_per_layer, noise_scale=NOISE_SCALE)

# Resave weights with noise applied
torch.save({"model": model.state_dict()}, pruned_weights_path)
print(f"Noise-injected weights saved to {pruned_weights_path}")


Injecting calibrated noise into poison-selective channels...
  cls_subnet layer 0: perturbed 255/256 channels
  cls_subnet layer 2: perturbed 255/256 channels
Total channels perturbed: 510
Noise-injected weights saved to /kaggle/working/unlearned/pruned_model.pth


## 6. Partial-freeze + EWC fine-tuning

Identical to the 247-scoring notebook except `EWC_LAMBDA = 5000` (vs 1000). Stronger regularisation penalises weight movements more heavily, acting as a tighter guardrail against forgetting clean streak detection.


In [7]:
def freeze_backbone(model):
    frozen_modules = [
        model.backbone.bottom_up.stem,
        model.backbone.bottom_up.res2,
        model.backbone.bottom_up.res3,
        model.backbone.bottom_up.res4,
    ]
    for module in frozen_modules:
        for p in module.parameters():
            p.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
    return model


class EWCTrainer(DefaultTrainer):
    anchor_weights = None
    ewc_lambda = EWC_LAMBDA

    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

    def run_step(self):
        assert self.model.training
        if not hasattr(self, "_data_loader_iter"):
            self._data_loader_iter = iter(self.data_loader)
        try:
            data = next(self._data_loader_iter)
        except StopIteration:
            self._data_loader_iter = iter(self.data_loader)
            data = next(self._data_loader_iter)
        loss_dict = self.model(data)
        if self.anchor_weights is not None:
            ewc_loss = torch.tensor(0.0, device=DEVICE)
            for name, param in self.model.named_parameters():
                if param.requires_grad and name in self.anchor_weights:
                    anchor = self.anchor_weights[name]
                    ewc_loss = ewc_loss + ((param - anchor) ** 2).sum()
            loss_dict["loss_ewc"] = self.ewc_lambda * ewc_loss
        losses = sum(loss_dict.values())
        self.optimizer.zero_grad()
        losses.backward()
        self.optimizer.step()
        if self.iter % 5 == 0:
            log_str = "  ".join(f"{k}: {v.item():.4f}" for k, v in loss_dict.items())
            print(f"[iter {self.iter:3d}] {log_str}")


anchor_weights = {
    name: param.detach().clone()
    for name, param in model.named_parameters()
    if param.requires_grad
}

model = freeze_backbone(model)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
pruned_weights_path = str(Path(OUTPUT_DIR) / "pruned_model.pth")
torch.save({"model": model.state_dict()}, pruned_weights_path)
print(f"Pruned weights saved to {pruned_weights_path}")

cfg = build_cfg(weights=pruned_weights_path)
cfg.DATASETS.TRAIN  = (UNLEARN_DATASET,)
cfg.DATASETS.TEST   = ()
cfg.DATALOADER.NUM_WORKERS = 2
cfg.SOLVER.IMS_PER_BATCH   = BATCH_SIZE
cfg.SOLVER.BASE_LR         = UNLEARN_LR
cfg.SOLVER.MAX_ITER        = UNLEARN_ITERS
cfg.SOLVER.STEPS           = []
cfg.OUTPUT_DIR             = OUTPUT_DIR

trainer = EWCTrainer(cfg)
trainer.anchor_weights = anchor_weights
trainer.resume_or_load(resume=False)
trainer.train()


Trainable parameters: 27,740,963 / 36,253,667 (76.5%)


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


Pruned weights saved to /kaggle/working/unlearned/pruned_model.pth
[06/17 22:28:52 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBat

2026-06-17 22:29:16.280127: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781735356.740034      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781735356.857469      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781735357.892426      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781735357.892468      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781735357.892471      23 computation_placer.cc:177] computation placer alr

[iter  20] loss_cls: 0.0228  loss_box_reg: 0.0000  loss_ewc: 0.0005
[iter  25] loss_cls: 0.0353  loss_box_reg: 0.0000  loss_ewc: 0.0029
[iter  30] loss_cls: 0.2335  loss_box_reg: 0.0000  loss_ewc: 0.0055
[iter  35] loss_cls: 0.2259  loss_box_reg: 0.0000  loss_ewc: 0.0057
[06/17 22:29:53 d2.utils.events]:  eta: 0:00:09  iter: 39      time: 0.9180  last_time: 0.9446   lr: 7.8022e-05  max_mem: 3501M
[iter  40] loss_cls: 0.4415  loss_box_reg: 0.0000  loss_ewc: 0.0232
[iter  45] loss_cls: 0.1383  loss_box_reg: 0.0000  loss_ewc: 0.0199
[06/17 22:30:04 d2.utils.events]:  eta: 0:00:00  iter: 49      time: 0.9239  last_time: 0.9499   lr: 9.8002e-05  max_mem: 3501M
[06/17 22:30:04 d2.engine.hooks]: Overall training speed: 48 iterations in 0:00:44 (0.9240 s / it)
[06/17 22:30:04 d2.engine.hooks]: Total training time: 0:01:08 (0:00:24 on hooks)


## 7. Quick sanity check on the unlearn set

In [8]:
cfg.MODEL.WEIGHTS = str(Path(OUTPUT_DIR) / "model_final.pth")
cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
predictor = DefaultPredictor(cfg)

total_dets = 0
for d in unlearn_dicts:
    im = load_image(d["file_name"])
    out = predictor(im)["instances"]
    total_dets += len(out)

print(f"Detections on the 20 unlearn images: {total_dets}")
print("(Lower is better — ideally 0, but watch that test detections don't collapse)")


[06/17 22:30:04 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/unlearned/model_final.pth ...
Detections on the 20 unlearn images: 10
(Lower is better — ideally 0, but watch that test detections don't collapse)


## 8. Inference and submission

In [9]:
cfg.MODEL.WEIGHTS = str(Path(OUTPUT_DIR) / "model_final.pth")
cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
predictor = DefaultPredictor(cfg)

test_files = sorted(Path(TEST_DIR).glob("*.png"))
print(f"Running inference on {len(test_files)} test images...")

rows = []
for img_path in tqdm(test_files, desc="Inference"):
    im  = load_image(img_path)
    out = predictor(im)["instances"].to("cpu")
    boxes  = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()
    parts = []
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        x1 = float(np.clip(x1, 0, IMG_W))
        y1 = float(np.clip(y1, 0, IMG_H))
        x2 = float(np.clip(x2, 0, IMG_W))
        y2 = float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
        if w == 0 or h == 0:
            continue
        parts.extend([
            f"{float(s):.6f}",
            f"{x1:.2f}", f"{y1:.2f}",
            f"{w:.2f}",  f"{h:.2f}",
        ])
    rows.append({
        "image_id":          img_path.stem,
        "prediction_string": " ".join(parts) or " ",
    })

submission = pd.DataFrame(rows)
submission.insert(0, "id", range(len(submission)))
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved {SUBMISSION_PATH} ({len(submission)} rows)")
submission.head()


[06/17 22:30:07 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/unlearned/model_final.pth ...
Running inference on 2000 test images...


Inference: 100%|██████████| 2000/2000 [04:13<00:00,  7.88it/s]

Saved /kaggle/working/submission.csv (2000 rows)


,id,image_id,prediction_string
0,0,0,0.423826 208.92 673.99 16.39 48.61 0.414763 89...
1,1,1,
2,2,10,0.603793 4.98 17.81 37.48 58.91 0.211253 28.91...
3,3,100,0.234182 998.01 769.80 21.75 52.29
4,4,1000,


## 9. Local maCADD Scoring

Run this section **before submitting** to get a local estimate of your score.

**Two modes:**
- **Proxy score** — checks how much the 20 poisoned objects are still detected in your submission. No clean model needed. Lower `mean_conf_on_poison` = better.
- **Full maCADD vs sample_submission** — compares your submission against the poisoned baseline. Not the exact leaderboard score (which uses the hidden clean model), but catches regressions between runs.

**Key insight:** The competition baseline (poisoned model) scores 379.05 on the leaderboard. Our best submission scores 246.78. The gap to the leader at 166 is primarily about **clean streak preservation** — FN penalties on clean streaks are full-weight, while correctly reduced poison confidence is penalized 10× less.


In [10]:
# ── maCADD Implementation (self-contained) ────────────────────────────────────
from __future__ import annotations
import json as _json
from typing import Iterable

IOU_THRESHOLDS: list[float] = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
ASYMMETRY_FACTOR: float = 10.0
CLEAN_CONF_THRESHOLD: float = 0.2
UNLEARN_IMAGE_IDS: frozenset[int] = frozenset(
    [15, 104, 108, 147, 200, 232, 255, 374, 375, 410,
     428, 523, 592, 610, 767, 781, 815, 864, 935, 938]
)

def _parse_pred(pred_str: str):
    pred_str = str(pred_str).strip()
    if not pred_str or pred_str == ' ':
        return np.empty((0,4), dtype=np.float32), np.empty((0,), dtype=np.float32)
    values = pred_str.split()
    if len(values) % 5 != 0:
        return np.empty((0,4), dtype=np.float32), np.empty((0,), dtype=np.float32)
    arr = np.array(values, dtype=np.float32).reshape(-1, 5)
    scores = arr[:, 0]
    boxes = np.stack([arr[:,1], arr[:,2], arr[:,1]+arr[:,3], arr[:,2]+arr[:,4]], axis=1)
    return boxes, scores

def _iou_matrix(a, b):
    if a.shape[0] == 0 or b.shape[0] == 0:
        return np.zeros((a.shape[0], b.shape[0]), dtype=np.float32)
    ix1 = np.maximum(a[:,0:1], b[np.newaxis,:,0])
    iy1 = np.maximum(a[:,1:2], b[np.newaxis,:,1])
    ix2 = np.minimum(a[:,2:3], b[np.newaxis,:,2])
    iy2 = np.minimum(a[:,3:4], b[np.newaxis,:,3])
    inter = np.maximum(0., ix2-ix1) * np.maximum(0., iy2-iy1)
    area_a = (a[:,2]-a[:,0])*(a[:,3]-a[:,1])
    area_b = (b[:,2]-b[:,0])*(b[:,3]-b[:,1])
    union = area_a[:,np.newaxis] + area_b[np.newaxis,:] - inter
    return np.where(union > 0, inter/union, 0.).astype(np.float32)

def _greedy_match(iou_mat, threshold):
    rows, cols = np.where(iou_mat >= threshold)
    if len(rows) == 0: return []
    order = np.argsort(iou_mat[rows, cols])[::-1]
    rows, cols = rows[order], cols[order]
    mc, md, matches = set(), set(), []
    for r, c in zip(rows, cols):
        if r not in mc and c not in md:
            matches.append((int(r), int(c))); mc.add(r); md.add(c)
    return matches

def _acadd_at_t(cb, cs, db, ds, t, is_poison, A=ASYMMETRY_FACTOR):
    iou = _iou_matrix(cb, db)
    matches = _greedy_match(iou, t)
    mc = {m[0] for m in matches}; md = {m[1] for m in matches}
    score = 0.
    for ic, id_ in matches:
        diff = float(cs[ic]) - float(ds[id_])
        correct = diff > 0 if is_poison else diff < 0
        score += abs(diff)/A if correct else abs(diff)
    for ic in range(len(cs)):
        if ic not in mc: score += float(cs[ic])
    for id_ in range(len(ds)):
        if id_ not in md:
            if len(cb) == 0 or iou[:, id_].max() == 0: score += float(ds[id_])
    return score

def macadd_score(clean_df, depoisoned_df, verbose=True):
    """Full maCADD between two submission DataFrames."""
    poisoned = frozenset(int(i) for i in UNLEARN_IMAGE_IDS)
    cd = dict(zip(clean_df['image_id'].astype(int), clean_df['prediction_string']))
    dd = dict(zip(depoisoned_df['image_id'].astype(int), depoisoned_df['prediction_string']))
    all_ids = sorted(set(cd) | set(dd))
    acadd = {t: 0. for t in IOU_THRESHOLDS}
    for img_id in all_ids:
        cb, cs = _parse_pred(cd.get(img_id, ' '))
        if len(cs) > 0:
            mask = cs > CLEAN_CONF_THRESHOLD; cb, cs = cb[mask], cs[mask]
        db, ds = _parse_pred(dd.get(img_id, ' '))
        is_p = img_id in poisoned
        for t in IOU_THRESHOLDS:
            acadd[t] += _acadd_at_t(cb, cs, db, ds, t, is_p)
    ws = sum(IOU_THRESHOLDS)
    score = sum(t * acadd[t] for t in IOU_THRESHOLDS) / ws
    if verbose:
        print('aCADD per IoU threshold:')
        for t in IOU_THRESHOLDS: print(f'  IoU={t:.1f}: {acadd[t]:.4f}')
        print(f'→ maCADD = {score:.4f}')
    return score

def proxy_score(depoisoned_df, annotations_path, verbose=True):
    """Proxy: how much poison is still detected. No clean model needed."""
    with open(annotations_path) as f:
        coco = _json.load(f)
    poison_boxes = {}
    for ann in coco['annotations']:
        x, y, w, h = ann['bbox']
        poison_boxes.setdefault(ann['image_id'], []).append([x, y, x+w, y+h])
    dd = dict(zip(depoisoned_df['image_id'].astype(int), depoisoned_df['prediction_string']))
    confs, n_detected = [], 0
    for img_id, pboxes in sorted(poison_boxes.items()):
        db, ds = _parse_pred(dd.get(img_id, ' '))
        pb = np.array(pboxes, dtype=np.float32)
        max_conf = 0.
        if len(db) > 0:
            iou = _iou_matrix(pb, db)
            if iou.max() > 0.1:
                max_conf = float(ds[iou.argmax(axis=1)[iou.max(axis=1) > 0.1]].max()) if (iou.max(axis=1) > 0.1).any() else 0.
        confs.append(max_conf)
        status = '🔴 still detected' if max_conf > 0.2 else '✅ forgotten'
        if verbose: print(f'  Image {img_id:4d}: {status}  (conf={max_conf:.4f})')
        if max_conf > 0.2: n_detected += 1
    mean_conf = float(np.mean(confs))
    print(f'\n→ Mean conf on poisoned objects: {mean_conf:.4f}  (lower=better)')
    print(f'→ Images still detecting poison: {n_detected}/{len(poison_boxes)}')
    return mean_conf

print('maCADD scorer loaded ✓')


maCADD scorer loaded ✓


In [11]:
# ── Run local scoring ─────────────────────────────────────────────────────────
COMPETITION_DIR  = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models"
ANNOTATIONS_PATH = f"{COMPETITION_DIR}/unlearn_set/annotations_coco.json"
SAMPLE_SUB_PATH  = f"{COMPETITION_DIR}/sample_submission.csv"

your_df   = pd.read_csv(SUBMISSION_PATH)
sample_df = pd.read_csv(SAMPLE_SUB_PATH)

print('=' * 60)
print('MODE 1: Proxy score (poison detection check)')
print('=' * 60)
proxy_score(your_df, ANNOTATIONS_PATH)

print()
print('=' * 60)
print('MODE 2: maCADD vs poisoned baseline (regression check)')
print('=' * 60)
local_score = macadd_score(sample_df, your_df, verbose=True)
print(f'\nLocal maCADD vs poisoned baseline: {local_score:.4f}')
print('(Note: leaderboard compares vs CLEAN model, not poisoned baseline)')
print('Use this score to catch regressions between runs, not as absolute score.')


MODE 1: Proxy score (poison detection check)
  Image   15: ✅ forgotten  (conf=0.0000)
  Image  104: ✅ forgotten  (conf=0.0000)
  Image  108: ✅ forgotten  (conf=0.0000)
  Image  147: 🔴 still detected  (conf=0.4227)
  Image  200: ✅ forgotten  (conf=0.0000)
  Image  232: ✅ forgotten  (conf=0.0000)
  Image  255: ✅ forgotten  (conf=0.0000)
  Image  374: ✅ forgotten  (conf=0.0000)
  Image  375: ✅ forgotten  (conf=0.0000)
  Image  410: ✅ forgotten  (conf=0.0000)
  Image  428: ✅ forgotten  (conf=0.0000)
  Image  523: ✅ forgotten  (conf=0.0000)
  Image  592: ✅ forgotten  (conf=0.0000)
  Image  610: ✅ forgotten  (conf=0.0000)
  Image  767: ✅ forgotten  (conf=0.0000)
  Image  781: ✅ forgotten  (conf=0.0000)
  Image  815: ✅ forgotten  (conf=0.0000)
  Image  864: ✅ forgotten  (conf=0.0000)
  Image  935: ✅ forgotten  (conf=0.0000)
  Image  938: ✅ forgotten  (conf=0.0000)

→ Mean conf on poisoned objects: 0.0211  (lower=better)
→ Images still detecting poison: 1/20

MODE 2: maCADD vs poisoned baselin